## 1. Kurulum

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, time, zipfile, subprocess

BASE = '/content/drive/MyDrive/bioinformatik'   # kalıcı klasör (Drive)
RAW  = '/content/raw'                            # geçici: ham veri
TRIM = '/content/trimmed'                        # geçici: temizlenmiş veri
run  = 'SRR2960032'                              # test örneği (inek 3141, enfekte çeyrek)
QC1  = f'{BASE}/results/fastqc_raw'
QC2  = f'{BASE}/results/fastqc_trimmed'
LOGS = f'{BASE}/results/trim_logs'
for p in [RAW, TRIM, QC1, QC2, LOGS]:
    os.makedirs(p, exist_ok=True)

def sh(cmd, show=True):
    """Komutu çalıştırır; hata olursa çıktıyı yazıp durur."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable='/bin/bash')
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        print(out)
        raise RuntimeError('komut başarısız: ' + cmd[:100])
    if show and out:
        print(out[-3000:])
    return out

# Araçlar: SRA Toolkit (indirme), pigz (sıkıştırma), FastQC, Trimmomatic
sh('apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y sra-toolkit pigz fastqc trimmomatic > /dev/null 2>&1', show=False)
print(sh('fasterq-dump --version | head -3'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fasterq-dump : 3.0.3 ( 3.0.2 )
fasterq-dump : 3.0.3 ( 3.0.2 )


## 2. Veri indirme (test örneği)
`prefetch` örneği NCBI SRA'dan indirir, `fasterq-dump` FASTQ dosyalarına çevirir, `pigz` sıkıştırır. Dosyalar zaten varsa atlanır.

In [ ]:
def ensure_sample(run):
    f1, f2 = f'{RAW}/{run}_1.fastq.gz', f'{RAW}/{run}_2.fastq.gz'
    if os.path.exists(f1) and os.path.exists(f2):
        print(run, ': ham dosyalar zaten var'); return f1, f2
    print(run, ': indiriliyor (6-10 dk)...')
    sh(f'prefetch {run} --max-size 50G -O {RAW}', show=False)
    sh(f'fasterq-dump {RAW}/{run}/{run}.sra --split-files --threads 4 -O {RAW}', show=False)
    sh(f'pigz -p 4 {RAW}/{run}_1.fastq {RAW}/{run}_2.fastq', show=False)
    sh(f'rm -rf {RAW}/{run}', show=False)   # ara .sra dosyasını sil
    print(run, ': hazır')
    return f1, f2

f1, f2 = ensure_sample(run)
print(sh(f'ls -lh {RAW}'))
# Kontrol: ilk okuma ve satır sayısı (her okuma 4 satır)
print(sh(f'zcat {f1} | head -4'))
n = int(sh(f'zcat {f1} | wc -l', show=False))
print('satır sayısı:', n, '-> okuma çifti:', n // 4)

SRR2960032 : ham dosyalar zaten var
total 3.9G
-rw-r--r-- 1 root root 2.0G Oct 19 19:09 SRR2960032_1.fastq.gz
-rw-r--r-- 1 root root 2.0G Oct 19 19:08 SRR2960032_2.fastq.gz
total 3.9G
-rw-r--r-- 1 root root 2.0G Oct 19 19:09 SRR2960032_1.fastq.gz
-rw-r--r-- 1 root root 2.0G Oct 19 19:08 SRR2960032_2.fastq.gz
@SRR2960032.1 1 length=101
NGAATGGTTCTTTTTTTCCTGAGTAGTAGGTCACGATATGAGAGATTATTCCAAACCCAGGTAAGATTAAAATATAGACTTCGGGGTGTCCAAAGAATCAG
+SRR2960032.1 1 length=101
#1:BDDFDHHHHHJJJJHHIJFHGGGHIIJHHIJJIIGHIJJIIHIJJIJHGIJIJJHIJJHFEEEHFFFDDFEEEEEEEDDDDDD;<<CDDDCCCDDDDC
@SRR2960032.1 1 length=101
NGAATGGTTCTTTTTTTCCTGAGTAGTAGGTCACGATATGAGAGATTATTCCAAACCCAGGTAAGATTAAAATATAGACTTCGGGGTGTCCAAAGAATCAG
+SRR2960032.1 1 length=101
#1:BDDFDHHHHHJJJJHHIJFHGGGHIIJHHIJJIIGHIJJIIHIJJIJHGIJIJJHIJJHFEEEHFFFDDFEEEEEEEDDDDDD;<<CDDDCCCDDDDC
satır sayısı: 101000568 -> okuma çifti: 25250142


## 3. FastQC (ham veri)
Okumaların kalitesini, adaptör kalıntısını ve diğer özellikleri raporlar. Raporlar Drive'a kaydedilir.

In [ ]:
f1, f2 = ensure_sample(run)
sh(f'fastqc -t 2 -o {QC1} {f1} {f2}', show=False)
print(sh(f'ls -lh {QC1}'))

def fastqc_ozet(qc_dir, ad):
    """FastQC zip dosyasından PASS/WARN/FAIL özetini yazdırır."""
    z = zipfile.ZipFile(f'{qc_dir}/{ad}_fastqc.zip')
    print(f'--- {ad} ---')
    print(z.read(f'{ad}_fastqc/summary.txt').decode())
    return z

for ad in [f'{run}_1', f'{run}_2']:
    fastqc_ozet(QC1, ad)

SRR2960032 : ham dosyalar zaten var
total 2.0M
-rw------- 1 root root 612K Oct 19 20:55 SRR2960032_1_fastqc.html
-rw------- 1 root root 402K Oct 19 20:55 SRR2960032_1_fastqc.zip
-rw------- 1 root root 587K Oct 19 20:55 SRR2960032_2_fastqc.html
-rw------- 1 root root 399K Oct 19 20:55 SRR2960032_2_fastqc.zip
total 2.0M
-rw------- 1 root root 612K Oct 19 20:55 SRR2960032_1_fastqc.html
-rw------- 1 root root 402K Oct 19 20:55 SRR2960032_1_fastqc.zip
-rw------- 1 root root 587K Oct 19 20:55 SRR2960032_2_fastqc.html
-rw------- 1 root root 399K Oct 19 20:55 SRR2960032_2_fastqc.zip
--- SRR2960032_1 ---
PASS	Basic Statistics	SRR2960032_1.fastq.gz
PASS	Per base sequence quality	SRR2960032_1.fastq.gz
PASS	Per sequence quality scores	SRR2960032_1.fastq.gz
WARN	Per base sequence content	SRR2960032_1.fastq.gz
PASS	Per sequence GC content	SRR2960032_1.fastq.gz
PASS	Per base N content	SRR2960032_1.fastq.gz
PASS	Sequence Length Distribution	SRR2960032_1.fastq.gz
FAIL	Sequence Duplication Levels	SRR296

In [ ]:
# Adaptör kalıntısı ve en sık geçen diziler (ham veri)
for ad in [f'{run}_1', f'{run}_2']:
    z = zipfile.ZipFile(f'{QC1}/{ad}_fastqc.zip')
    txt = z.read(f'{ad}_fastqc/fastqc_data.txt').decode()
    for mod in ['Overrepresented sequences', 'Adapter Content']:
        s = txt.index('>>' + mod); e = txt.index('>>END_MODULE', s)
        lines = txt[s:e].splitlines()
        print(f'--- {ad}: {mod} ---')
        print('\n'.join(lines[:8] if 'Over' in mod else lines[:3] + lines[-3:]))

--- SRR2960032_1: Overrepresented sequences ---
>>Overrepresented sequences	fail
#Sequence	Count	Percentage	Possible Source
GATCGGAAGAGCACACGTCTGAACTCCAGTCACGATCAGATCTCGTATGC	1821724	7.2147079410484105	TruSeq Adapter, Index 9 (100% over 50bp)
CGTATGCCGTCTTCTGCTTGAGATCGGAAGAGCACACGTCTGAACTCCAG	452235	1.7910196307014827	Illumina Multiplexing PCR Primer 2.01 (100% over 30bp)
AGATCGGAAGAGCACACGTCTGAACTCCAGTCACGATCAGATCTCGTATG	76827	0.30426363542826806	TruSeq Adapter, Index 9 (100% over 49bp)
CCAGCTTGCTGCTTCTTCCCAGTCTTGGGTTCAAGATCTTGACAACCATG	28672	0.11355183665897799	No Hit
CTTTGACTCAGTGGCCTTTATACCAGCATGATACTACTGGAATTTAGGTA	28378	0.11238748677136154	No Hit
--- SRR2960032_1: Adapter Content ---
>>Adapter Content	warn
#Position	Illumina Universal Adapter	Illumina Small RNA 3' Adapter	Illumina Small RNA 5' Adapter	Nextera Transposase Sequence	PolyA	PolyG
1	0.3532217759409036	7.920747534805943E-6	0.0	0.0	0.005132644402554251	5.54452327436416E-5
86-87	4.873214178359868	5.425712061342071E-4	3.96

## 4. Trimmomatic (temizleme)
Adaptörleri ve kalitesi düşük uçları keser.

- `ILLUMINACLIP:TruSeq3-PE-2.fa:2:30:10:2:True`: adaptör dizilerini bulup keser
- `LEADING:3`, `TRAILING:3`: başındaki ve sonundaki çok düşük kaliteli bazları atar
- `SLIDINGWINDOW:4:15`: 4 bazlık pencerede ortalama kalite 15'in altına inerse kalanını keser
- `MINLEN:36`: 36 bazdan kısa kalan okumaları atar

Not: Bu sistemde `trimmomatic` komutu tanımlı olmadığı için program doğrudan `java -jar` ile çalıştırılır.

In [ ]:
f1, f2 = ensure_sample(run)
jars = sorted(glob.glob('/usr/share/java/trimmomatic*.jar') + glob.glob('/usr/share/trimmomatic/*.jar'))
print('bulunan jar:', jars)
assert jars, 'Trimmomatic jar dosyası bulunamadı'
JAR   = jars[0]
ADAPT = '/usr/share/trimmomatic/TruSeq3-PE-2.fa'

t0 = time.time()
log = sh(f'java -jar {JAR} PE -threads 4 -phred33 {f1} {f2} '
         f'{TRIM}/{run}_1P.fastq.gz {TRIM}/{run}_1U.fastq.gz {TRIM}/{run}_2P.fastq.gz {TRIM}/{run}_2U.fastq.gz '
         f'ILLUMINACLIP:{ADAPT}:2:30:10:2:True LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36', show=False)
open(f'{LOGS}/{run}_trimmomatic.log', 'w').write(log)
print([l for l in log.splitlines() if 'Input Read Pairs' in l or 'Completed' in l])
print(f'süre: {(time.time()-t0)/60:.1f} dk')
print(sh(f'ls -lh {TRIM}'))

SRR2960032 : ham dosyalar zaten var
bulunan jar: ['/usr/share/java/trimmomatic-0.39.jar', '/usr/share/java/trimmomatic.jar']
['Input Read Pairs: 25250142 Both Surviving: 20889433 (82.73%) Forward Only Surviving: 1117975 (4.43%) Reverse Only Surviving: 272441 (1.08%) Dropped: 2970293 (11.76%)', 'TrimmomaticPE: Completed successfully']
süre: 28.6 dk
total 3.3G
-rw-r--r-- 1 root root 1.6G Oct 19 20:37 SRR2960032_1P.fastq.gz
-rw-r--r-- 1 root root  94M Oct 19 20:37 SRR2960032_1U.fastq.gz
-rw-r--r-- 1 root root 1.7G Oct 19 20:37 SRR2960032_2P.fastq.gz
-rw-r--r-- 1 root root  22M Oct 19 20:37 SRR2960032_2U.fastq.gz
total 3.3G
-rw-r--r-- 1 root root 1.6G Oct 19 20:37 SRR2960032_1P.fastq.gz
-rw-r--r-- 1 root root  94M Oct 19 20:37 SRR2960032_1U.fastq.gz
-rw-r--r-- 1 root root 1.7G Oct 19 20:37 SRR2960032_2P.fastq.gz
-rw-r--r-- 1 root root  22M Oct 19 20:37 SRR2960032_2U.fastq.gz


## 5. FastQC (temizlenmiş veri)
Temizlemenin işe yaradığını doğrularız. Beklenen: **Adapter Content PASS**.

In [ ]:
sh(f'fastqc -t 4 -o {QC2} {TRIM}/{run}_1P.fastq.gz {TRIM}/{run}_2P.fastq.gz', show=False)
for ad in [f'{run}_1P', f'{run}_2P']:
    fastqc_ozet(QC2, ad)

--- SRR2960032_1P ---
PASS	Basic Statistics	SRR2960032_1P.fastq.gz
PASS	Per base sequence quality	SRR2960032_1P.fastq.gz
PASS	Per sequence quality scores	SRR2960032_1P.fastq.gz
FAIL	Per base sequence content	SRR2960032_1P.fastq.gz
WARN	Per sequence GC content	SRR2960032_1P.fastq.gz
PASS	Per base N content	SRR2960032_1P.fastq.gz
WARN	Sequence Length Distribution	SRR2960032_1P.fastq.gz
FAIL	Sequence Duplication Levels	SRR2960032_1P.fastq.gz
WARN	Overrepresented sequences	SRR2960032_1P.fastq.gz
PASS	Adapter Content	SRR2960032_1P.fastq.gz

--- SRR2960032_2P ---
PASS	Basic Statistics	SRR2960032_2P.fastq.gz
PASS	Per base sequence quality	SRR2960032_2P.fastq.gz
PASS	Per sequence quality scores	SRR2960032_2P.fastq.gz
FAIL	Per base sequence content	SRR2960032_2P.fastq.gz
WARN	Per sequence GC content	SRR2960032_2P.fastq.gz
PASS	Per base N content	SRR2960032_2P.fastq.gz
WARN	Sequence Length Distribution	SRR2960032_2P.fastq.gz
FAIL	Sequence Duplication Levels	SRR2960032_2P.fastq.gz
WARN	Overrepres

## 6. DESeq2 girdisini hazırla
Bu bölüm test örneğinden bağımsızdır. GEO'da yazarların yayımladığı **hazır sayım tablosunu** (`GSE75379_count.txt.gz`) kullanır. `01_prepare_inputs.py` betiği 12 meme örneğini (6 inek, 24. saat) seçer ve sayıların tam sayı (ham sayım) olduğunu doğrular.

In [ ]:
CNT = f'{BASE}/GSE75379_count.txt.gz'
if not os.path.exists(CNT):
    os.system(f'wget -q -O {CNT} https://ftp.ncbi.nlm.nih.gov/geo/series/GSE75nnn/GSE75379/suppl/GSE75379_count.txt.gz')
if os.path.exists(CNT) and os.path.getsize(CNT) < 100000:
    os.remove(CNT)
assert os.path.exists(CNT), 'GSE75379_count.txt.gz bulunamadı: GEO sayfasından indirip Drive/bioinformatik klasörüne koyun'

def bul(desen):
    hit = glob.glob(f'{BASE}/**/{desen}', recursive=True)
    return hit[0] if hit else None

PREP, META, RSCR = bul('*prepare_inputs*.py'), bul('*udder_24h_metadata*.csv'), bul('*deseq2*.R')
print('prep:', PREP); print('meta:', META); print('R   :', RSCR)
assert None not in (PREP, META, RSCR), 'betik veya metadata dosyası Drive klasöründe bulunamadı'

OUT = f'{BASE}/results/deseq2'
print(sh(f'python3 "{PREP}" --counts {CNT} --meta "{META}" --out {OUT}'))

prep: /content/drive/MyDrive/bioinformatik/01_prepare_inputs.py
meta: /content/drive/MyDrive/bioinformatik/udder_24h_metadata.csv
R   : /content/drive/MyDrive/bioinformatik/02_deseq2.R
12 örnek, 6 inek seçildi
sayım tablosu: 24616 satır x 36 sütun
örnek başına toplam okuma (milyon):
A377-115    13.26
A377-116    27.00
A377-71     13.47
A377-72     15.77
A377-103    22.28
A377-104    32.11
A377-97     16.97
A377-98     13.37
A377-65     16.94
A377-66     11.53
A377-67     17.49
A377-68     17.57
yazıldı: /content/drive/MyDrive/bioinformatik/results/deseq2 -> counts.csv, coldata.csv
12 örnek, 6 inek seçildi
sayım tablosu: 24616 satır x 36 sütun
örnek başına toplam okuma (milyon):
A377-115    13.26
A377-116    27.00
A377-71     13.47
A377-72     15.77
A377-103    22.28
A377-104    32.11
A377-97     16.97
A377-98     13.37
A377-65     16.94
A377-66     11.53
A377-67     17.49
A377-68     17.57
yazıldı: /content/drive/MyDrive/bioinformatik/results/deseq2 -> counts.csv, coldata.csv


## 7. DESeq2
R ve DESeq2 kurulur, analiz çalıştırılır. Model: `~ inek + durum` (aynı ineğin iki çeyreği eşleştirilir). Çıktılar `results/deseq2` klasörüne yazılır: tüm sonuçlar, anlamlı genler, PCA, volcano ve ısı haritası.

In [ ]:
sh('apt-get -qq install -y r-bioc-deseq2 r-cran-ggplot2 r-cran-pheatmap > /dev/null 2>&1', show=False)
print(sh(f'Rscript "{RSCR}" {OUT}'))

Analize giren gen sayısı: 14369 


out of 14369 with nonzero total read count
adjusted p-value < 0.05
LFC > 0 (up)       : 2079, 14%
LFC < 0 (down)     : 1747, 12%
outliers [1]       : 0, 0%
low counts [2]     : 0, 0%
(mean count < 3)
[1] see 'cooksCutoff' argument of ?results
[2] see 'independentFiltering' argument of ?results


Anlamlı gen (padj < 0.05 ve |log2FC| >= 1): 1931  (artan: 1236  azalan: 695 )

Bitti. Çıktılar: /content/drive/MyDrive/bioinformatik/results/deseq2 
estimating size factors
estimating dispersions
gene-wise dispersion estimates
mean-dispersion relationship
final dispersion estimates
fitting model and testing
using ntop=500 top features by variance
Analize giren gen sayısı: 14369 


out of 14369 with nonzero total read count
adjusted p-value < 0.05
LFC > 0 (up)       : 2079, 14%
LFC < 0 (down)     : 1747, 12%
outliers [1]       : 0, 0%
low counts [2]     : 0, 0%
(mean count < 3)
[1] see 'cooksCutoff' argument of ?results
[2] see 'independentFiltering' argument o

## 8. Gen adları ve en önemli genler
Ensembl kimliklerini (ENSBTAG...) Ensembl REST ile gen adlarına çevirir. Bazı eski kimlikler için ad bulunamayabilir.

In [ ]:
import pandas as pd, requests, json
sig = pd.read_csv(f'{OUT}/deseq2_significant.csv')

def isimler(ids):
    out = {}
    for i in range(0, len(ids), 500):
        try:
            r = requests.post('https://rest.ensembl.org/lookup/id',
                              headers={'Content-Type': 'application/json', 'Accept': 'application/json'},
                              data=json.dumps({'ids': ids[i:i+500]}), timeout=60)
            if r.ok:
                for k, v in r.json().items():
                    if v: out[k] = (v.get('display_name'), (v.get('description') or '').split(' [')[0])
            else:
                print('Ensembl yanıtı:', r.status_code)
        except Exception as e:
            print('bağlantı hatası:', e); break
    return out

names = isimler(list(sig.gene))
sig['symbol']      = sig.gene.map(lambda g: names.get(g, (None, None))[0])
sig['description'] = sig.gene.map(lambda g: names.get(g, (None, None))[1])
sig.to_csv(f'{OUT}/deseq2_significant_annotated.csv', index=False)

cols = ['gene', 'symbol', 'baseMean', 'log2FoldChange', 'padj', 'description']
yuksek = sig[sig.baseMean >= 50]
print('En çok artan 15 (ortalama sayımı >= 50):')
print(yuksek.sort_values('log2FoldChange', ascending=False).head(15)[cols].to_string(index=False))
print('\nEn çok azalan 15:')
print(yuksek.sort_values('log2FoldChange').head(15)[cols].to_string(index=False))
print('\nEn anlamlı 15:')
print(sig.sort_values('padj').head(15)[cols].to_string(index=False))

En çok artan 15 (ortalama sayımı >= 50):
              gene   symbol      baseMean  log2FoldChange         padj                            description
ENSBTAG00000035183     None    554.666807       10.462695 2.078579e-10                                   None
ENSBTAG00000025531 PSORS1C2    363.617897        9.699585 1.066401e-12 psoriasis susceptibility 1 candidate 2
ENSBTAG00000001785     TGM3   3441.370498        8.827708 4.652741e-14                     transglutaminase 3
ENSBTAG00000004714   MCEMP1    488.836236        8.169927 2.995198e-12 mast cell expressed membrane protein 1
ENSBTAG00000026753    CXCR1    510.682340        8.096613 5.030280e-14     chemokine (C-X-C motif) receptor 1
ENSBTAG00000013334    CSF3R   3589.639209        7.937697 2.147480e-20   colony stimulating factor 3 receptor
ENSBTAG00000022395     None  14634.607723        7.845840 4.447912e-23                                       
ENSBTAG00000007901     None   1046.331406        7.757577 2.791780e-64         